# Stage 1 — Boundary Sensitivity Sweep
**MGSM Telugu evaluation with Qwen2.5-1.5B two-cut hard swap**

Run from the `stage1/` directory (or set `STAGE1_DIR` below).
```bash
# In Colab:
# !git clone <repo>
# %cd <repo>/stage1
```

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
import sys, os

# Ensure imports resolve from stage1/
STAGE1_DIR = os.path.abspath('.')
if STAGE1_DIR not in sys.path:
    sys.path.insert(0, STAGE1_DIR)

import gc
import torch
import pandas as pd

from utils.config    import load_config
from data.loader     import load_mgsm
from models.composer import load_models, get_condition_model
from inference.runner  import run_inference
from inference.parser  import parse_answer
from analysis.bds      import compute_bds, compute_bds_sweep
from analysis.evaluator import evaluate_all
from utils.logger    import (
    create_run_dir, save_results, save_hidden_states,
    save_bds_results, save_evaluation, save_manifest,
)

# ── Shared state ──────────────────────────────────────────────────────────────
_results_cache  = {}   # {condition_name: [parsed_result_dicts]}
_hidden_cache   = {}   # {condition_name: [{sample_id, hidden_states}]}
_metadata_cache = {}   # {condition_name: metadata dict from get_condition_model}
recipient = donor = tokenizer = None
run_dir = None
samples = None  # populated in Cell 3


def _get_or_load_base_models(cfg):
    """Load recipient + donor once; return from cache on subsequent calls."""
    global recipient, donor, tokenizer
    if recipient is None:
        print('Loading models (first call)...')
        recipient, donor, tokenizer = load_models(
            recipient_name=cfg.models.recipient,
            donor_name=cfg.models.donor,
        )
        print('  Models ready.')
    return recipient, donor, tokenizer


def run_condition(condition_name, cfg):
    """
    Load model, run inference, parse answers.
    Results are cached so cells can be re-run without full re-inference.
    Returns list of parsed result dicts (with gold_answer included).
    """
    global run_dir
    if run_dir is None:
        run_dir = create_run_dir(base_dir='outputs')
        print(f'Run directory: {run_dir}')

    if condition_name in _results_cache:
        print(f'{condition_name}: using cached results')
        return _results_cache[condition_name]

    rec, don, tok = _get_or_load_base_models(cfg)

    if condition_name.startswith('hard_swap_b'):
        cond_key = 'hard_swap'
        b = int(condition_name.split('_b')[1])
        t = cfg.t_fixed
    elif condition_name == 'reference':
        cond_key = 'reference'
        b = cfg.reference.b_ref
        t = cfg.reference.t_ref
    elif condition_name == 'random_donor':
        cond_key = 'random_donor'
        b = cfg.boundary_grid[0]
        t = cfg.t_fixed
    else:  # no_swap
        cond_key = 'no_swap'
        b = t = None

    model, cond_meta = get_condition_model(
        recipient=rec, donor=don,
        condition=cond_key, b=b, t=t,
        b_ref=cfg.reference.b_ref, t_ref=cfg.reference.t_ref,
        random_donor_seed=cfg.random_donor.seed,
    )
    _metadata_cache[condition_name] = cond_meta  # persist for manifest

    gen_cfg = {
        'do_sample':      cfg.generation.do_sample,
        'temperature':    cfg.generation.temperature,
        'max_new_tokens': cfg.generation.max_new_tokens,
    }
    inf_results = run_inference(
        model=model, tokenizer=tok, samples=samples,
        generation_config=gen_cfg, pooling=cfg.hidden_state.pooling,
    )

    parsed = []
    sample_map = {s['sample_id']: s for s in samples}
    for r in inf_results:
        p = parse_answer(r['output_text'])
        parsed.append({
            'sample_id':   r['sample_id'],
            'output_text': r['output_text'],
            'gold_answer': sample_map[r['sample_id']]['gold_answer'],
            **p,
        })

    hs_list = [
        {'sample_id': r['sample_id'], 'hidden_states': r['hidden_states']}
        for r in inf_results
    ]
    _hidden_cache[condition_name]  = hs_list
    _results_cache[condition_name] = parsed

    save_results(run_dir, condition_name, parsed)
    save_hidden_states(run_dir, condition_name, inf_results)

    if condition_name != 'no_swap':
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if 'source_start' in cond_meta:
        print(
            f'  random_donor: seed={cfg.random_donor.seed}  '
            f'source_start={cond_meta["source_start"]}'
        )

    return parsed


# ── Sanity checks ─────────────────────────────────────────────────────────────
print('=== Sanity Checks ===')
print('1. Hidden state extraction : prompt-only forward pass (fixed anchor)')
print('2. H1 uses degradation     : max(0, baseline - condition), not abs(delta)')
print('3. Criterion 1             : CI must be entirely negative (ci_hi < 0)')
print('4. Criterion 2             : bootstrap distribution of rank correlation (>80% positive)')
print('Config not loaded yet — seed and b_ref will be shown after Cell 2.')
print('Setup complete.')

In [ ]:
# ── Cell 2: Config ────────────────────────────────────────────────────────────
# Switch to stage1_sanity.yaml for a quick 5-sample debug run:
#   config = load_config('configs/stage1_sanity.yaml')

config = load_config('configs/stage1_main.yaml')

# Optional debug override (comment out for full run):
# config.dataset.debug_n = 5

print(f'Recipient     : {config.models.recipient}')
print(f'Donor         : {config.models.donor}')
print(f'Boundary grid : {config.boundary_grid}')
print(f't_fixed       : {config.t_fixed}')
print(f'Reference     : b_ref={config.reference.b_ref}, t_ref={config.reference.t_ref}')
print(f'Pooling       : {config.hidden_state.pooling}')
print(f'Random seed   : {config.random_donor.seed}')
print(f'Dataset       : {config.dataset.name} lang={config.dataset.lang} split={config.dataset.split}')
print(f'debug_n       : {config.dataset.debug_n}')
print()
print('5. Random donor seed      :', config.random_donor.seed)
print('6. Reference b_ref fixed  :', config.reference.b_ref)

In [ ]:
# ── Cell 3: Data ──────────────────────────────────────────────────────────────
samples = load_mgsm(
    dataset_name=config.dataset.name,
    lang=config.dataset.lang,
    split=config.dataset.split,
    debug_n=config.dataset.debug_n,
)
print(f'Loaded {len(samples)} samples')
pd.DataFrame(samples[:3])[['sample_id', 'gold_answer', 'prompt']].assign(
    prompt=lambda df: df.prompt.str[:80] + '...'
)

In [ ]:
# ── Cell 4: No-swap baseline ──────────────────────────────────────────────────
no_swap_results = run_condition('no_swap', config)

n_ok = sum(r['parse_success'] for r in no_swap_results)
print(f'Parse success: {n_ok}/{len(samples)}')
pd.DataFrame(no_swap_results[:5])[['sample_id', 'gold_answer', 'parsed_answer', 'parse_success']]

In [ ]:
# ── Cell 5: Hard swap sweep ───────────────────────────────────────────────────
sweep_results = {}
for b in config.boundary_grid:
    cond = f'hard_swap_b{b}'
    print(f'Running {cond}...')
    sweep_results[cond] = run_condition(cond, config)
    n_ok = sum(r['parse_success'] for r in sweep_results[cond])
    print(f'  Done. Parse success: {n_ok}/{len(samples)}')

print('Sweep complete.')

In [ ]:
# ── Cell 6: Reference & control ───────────────────────────────────────────────
ref_results  = run_condition('reference',    config)
ctrl_results = run_condition('random_donor', config)

print(f'Reference parse success    : {sum(r["parse_success"] for r in ref_results)}/{len(samples)}')
print(f'Random donor parse success : {sum(r["parse_success"] for r in ctrl_results)}/{len(samples)}')
print(f'Random donor seed: {config.random_donor.seed}')

In [ ]:
# ── Cell 7: BDS ───────────────────────────────────────────────────────────────
no_swap_hidden = _hidden_cache['no_swap']

sweep_hidden = {
    cond: _hidden_cache[cond]
    for cond in list(sweep_results.keys()) + ['reference', 'random_donor']
    if cond in _hidden_cache
}

bds_results = compute_bds_sweep(
    hidden_no_swap=no_swap_hidden,
    hidden_sweep=sweep_hidden,
    config=config,
)

for cond_name, bds in bds_results.items():
    save_bds_results(run_dir, cond_name, bds)

rows = []
for cond, bds in bds_results.items():
    agg = bds['aggregate']
    rows.append({
        'condition':     cond,
        'b':             bds['b'],
        'bds_lower':     round(agg['mean_bds_lower'],  4),
        'bds_upper':     round(agg['mean_bds_upper'],  4),
        'bds_total':     round(agg['mean_bds_total'],  4),
        'cka_bds_lower': round(agg['cka_bds_lower'],   4),
        'cka_bds_upper': round(agg['cka_bds_upper'],   4),
        'cka_bds_total': round(agg['cka_bds_total'],   4),
    })
pd.DataFrame(rows).set_index('condition')

In [ ]:
# ── Cell 8: Evaluation ────────────────────────────────────────────────────────
eval_results = evaluate_all(
    no_swap=no_swap_results,
    sweep=sweep_results,
    reference=ref_results,
    control=ctrl_results,
    bds=bds_results,
    config=config,
    samples=samples,
)

save_evaluation(run_dir, eval_results.__dict__)

print('=== Systematic Criteria ===')
print(f'Criterion 1 (delta CI entirely negative) : {eval_results.criterion_1}')
print(f'Criterion 2 (bootstrap positive rate)    : {eval_results.criterion_2_rate:.3f}  → passes={eval_results.criterion_2}')
print(f'Criterion 3 (ordering stability)         : {eval_results.criterion_3}')
n_met = sum([eval_results.criterion_1, eval_results.criterion_2, eval_results.criterion_3])
print(f'Systematic H1 support: {n_met}/3 criteria met  (threshold=2)  → {n_met >= 2}')

In [ ]:
# ── Cell 9: Results Summary ───────────────────────────────────────────────────
all_conditions_ordered = (
    ['no_swap']
    + [f'hard_swap_b{b}' for b in config.boundary_grid]
    + ['reference', 'random_donor']
)

rows = []
for cond in all_conditions_ordered:
    acc_info = eval_results.accuracies.get(cond, {})
    bds_info = bds_results.get(cond, {}).get('aggregate', {})
    rows.append({
        'condition':      cond,
        'accuracy':       round(acc_info.get('accuracy',         0.0), 4),
        'valid_accuracy': round(acc_info.get('valid_accuracy',   0.0), 4),
        'delta_vs_nosw':  round(eval_results.deltas.get(cond,    0.0), 4),
        'valid_rate':     round(acc_info.get('valid_answer_rate', 0.0), 4),
        'bds_lower':      round(bds_info.get('mean_bds_lower',   float('nan')), 4),
        'bds_upper':      round(bds_info.get('mean_bds_upper',   float('nan')), 4),
        'bds_total':      round(bds_info.get('mean_bds_total',   float('nan')), 4),
        'cka_bds_total':  round(bds_info.get('cka_bds_total',    float('nan')), 4),
    })

summary = pd.DataFrame(rows).set_index('condition')
display(summary)

print(f'\nH1 BDS-degradation rank correlation:')
print(f'  rho = {eval_results.bds_delta_rho}')
print(f'  p   = {eval_results.bds_delta_p}')
if eval_results.bds_delta_ci:
    print(f'  bootstrap CI = {[round(v, 4) for v in eval_results.bds_delta_ci]}')

# Manifest — source_start logged from _metadata_cache populated by run_condition()
rand_meta = _metadata_cache.get('random_donor', {})
rand_src  = rand_meta.get('source_start')  # integer or None
print(f'\nrandom_donor source_start: {rand_src}')

no_swap_hs = _hidden_cache.get('no_swap', [])
hidden_state_info = None
if no_swap_hs:
    hs0 = no_swap_hs[0]['hidden_states']
    hidden_state_info = {
        'layer_count': hs0.shape[0],
        'shape':       list(hs0.shape),
        'dtype':       str(hs0.dtype),
    }

save_manifest(
    run_dir=run_dir,
    config=config,
    conditions=all_conditions_ordered,
    sample_ids=[s['sample_id'] for s in samples],
    hidden_state_info=hidden_state_info,
    random_donor_source_start=rand_src,
)
print(f'Manifest saved to: {run_dir}/manifest.json')